# BƯỚC 1: CÀI ĐẶT

In [ ]:
print("📦 Đang cài đặt Earth Engine...")
!pip install earthengine-api --quiet

# BƯỚC 2: IMPORT


In [ ]:
import ee
import time
from datetime import datetime

# BƯỚC 3: XÁC THỰC

In [ ]:
##  PROJECT ID (thay bằng project của bạn)
PROJECT_ID = 'steam-spider-478112-g2'  # ← VD: 'hidden-talon-429103-k9'

In [ ]:

print("\n🔐 Đang xác thực Earth Engine...")
print("⚠️  Lần đầu chạy: làm theo hướng dẫn để đăng nhập")

try:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)
    print("✅ Đã khởi tạo Earth Engine thành công!")
except Exception as e:
    print(f"❌ Lỗi: {e}")
    print("\n💡 Vui lòng:")
    print(f"   1. Thay PROJECT_ID = '{PROJECT_ID}'")
    print("   2. Bằng project ID thực của bạn")
    raise

# BƯỚC 4: LẤY RANH GIỚI VIỆT NAM

In [ ]:
countries = ee.FeatureCollection("FAO/GAUL/2015/level0")
vietnam = countries.filter(ee.Filter.eq('ADM0_NAME', 'Viet Nam')).first()
vietnam_geometry = vietnam.geometry()
print("✅ Đã load ranh giới Việt Nam")

# BƯỚC 5: ĐỊNH NGHĨA HÀM XỬ LÝ

In [ ]:
# 1. Hàm xử lý band MOD16 với mask chất lượng và scale factor 
def process_mod16_band(data_band, qc_band, new_name, scale_factor):
    modland_qc = qc_band.bitwiseAnd(1)
    good_quality_mask = modland_qc.eq(0)
    valid_data_mask = data_band.lte(32700)
    combined_mask = good_quality_mask.And(valid_data_mask)

    return data_band.updateMask(combined_mask).multiply(scale_factor).rename(new_name)

# 2. Hàm export ảnh lên Google Drive
def export_to_drive(image, date_string, folder_name):
    output_name = f'MOD16A2GF_Vietnam_ET_PET_{date_string}'

    task = ee.batch.Export.image.toDrive(
        image=image,
        description=output_name,
        folder=folder_name,
        fileNamePrefix=output_name,
        region=vietnam_geometry.bounds(),
        scale=500,
        crs='EPSG:4326',
        maxPixels=1e13,
        fileFormat='GeoTIFF'
    )

    task.start()
    print(f'✓ Đã tạo task: {output_name}')
    return task

# 3. Hàm xử lý dữ liệu MOD16 cho một ngày cụ thể
def process_date(date_string, folder_name):
    print(f'\n🔄 Bắt đầu xử lý cho ngày: {date_string}')

    ## Tạo đối tượng ngày
    target_date = ee.Date(date_string)

    ## Lấy collection MOD16A2GF
    mod16_collection = ee.ImageCollection('MODIS/061/MOD16A2GF')

    ## Tìm ảnh gần nhất trước hoặc bằng ngày target
    potential_images = mod16_collection.filter(
        ee.Filter.lte('system:time_start', target_date.millis())
    ).sort('system:time_start', False).limit(1)

    ## Kiểm tra số lượng ảnh
    image_count = potential_images.size().getInfo()
    print(f'Số ảnh MOD16A2GF tìm thấy: {image_count}')

    ## Nếu không có ảnh, thông báo và thoát
    if image_count == 0:
        print(f'⚠️  Không có dữ liệu MOD16 cho ngày {date_string}')
        return False

    ## Lấy ảnh đầu tiên trong collection (ảnh gần nhất)
    image_mod16 = potential_images.first()

    ## Clip theo ranh giới Việt Nam
    clipped_image = image_mod16.clip(vietnam_geometry)

    ## Lấy các band
    et_raw = clipped_image.select('ET')
    le_raw = clipped_image.select('LE')
    pet_raw = clipped_image.select('PET')
    ple_raw = clipped_image.select('PLE')
    et_qc = clipped_image.select('ET_QC')

    ## Xử lý các band với mask chất lượng
    print('Đang áp dụng mask chất lượng...')
    et = process_mod16_band(et_raw, et_qc, 'ET', 0.1)
    le = process_mod16_band(le_raw, et_qc, 'LE', 10000)
    pet = process_mod16_band(pet_raw, et_qc, 'PET', 0.1)
    ple = process_mod16_band(ple_raw, et_qc, 'PLE', 10000)

    ## Kết hợp các band
    final_image = ee.Image.cat([et, le, pet, ple]).toFloat()

    ## Export lên Drive
    try:
        export_to_drive(final_image, date_string, folder_name)
        return True
    except Exception as e:
        print(f'✗ Lỗi: {str(e)}')
        return False
    
# 4. Hàm kiểm tra trạng thái các task
def check_tasks_status():
    """Kiểm tra trạng thái các task"""
    tasks = ee.batch.Task.list()
    status_count = {
        'READY': 0, 'RUNNING': 0, 'COMPLETED': 0,
        'FAILED': 0, 'CANCELLED': 0
    }

    for task in tasks[:100]:
        state = task.status()['state']
        if state in status_count:
            status_count[state] += 1

    return status_count

# BƯỚC 6:DANH SÁCH NGÀY CHO TỪNG NĂM


In [ ]:
# CHỌN NĂM CẦN XỬ LÝ
YEAR = 2024  # ← ĐỔI THÀNH: 2020, 2021, 2022, 2023, 2024

# CHỌN TÊN FOLDER (tùy chọn)
FOLDER_BASE_NAME = 'mod16_et_pet'  # ← Có thể đổi thành: 'modis_et', 'et_vietnam'...

# ============================================================
# 📅 DANH SÁCH NGÀY CHO TỪNG NĂM
# ============================================================

dates_by_year = {
    # Năm 2020 - ĐÃ CÓ SẴN
    2020: [
        '2020-01-08','2020-01-16','2020-01-18','2020-01-30',
        '2020-02-12','2020-02-19','2020-02-22','2020-02-24',
        '2020-02-26','2020-03-04','2020-03-10','2020-03-22',
        '2020-03-26','2020-03-29','2020-03-30','2020-04-03',
        '2020-04-05','2020-04-12','2020-04-18','2020-04-21',
        '2020-04-28','2020-04-30','2020-05-04','2020-05-05',
        '2020-05-07','2020-06-01','2020-06-05','2020-06-12',
        '2020-06-20','2020-07-05','2020-07-17','2020-07-26',
        '2020-07-29','2020-08-04','2020-08-18','2020-08-20',
        '2020-08-24','2020-08-25','2020-08-26','2020-08-27',
        '2020-08-30','2020-09-05','2020-09-22','2020-09-30',
        '2020-10-20','2020-10-30','2020-11-07','2020-11-08',
        '2020-11-23','2020-11-26','2020-11-27','2020-11-30',
        '2020-12-06','2020-12-12','2020-12-25','2020-12-29'
    ],

    # Năm 2021 - THÊM NGÀY VÀO ĐÂY
    2021: [
        '2021-01-02','2021-01-16','2021-01-18','2021-01-26',
        '2021-01-28','2021-02-05','2021-02-10','2021-02-12',
        '2021-02-17','2021-02-26','2021-03-01','2021-03-05',
        '2021-03-06','2021-03-07','2021-03-14','2021-03-16',
        '2021-03-22','2021-03-30','2021-04-19','2021-04-24',
        '2021-04-26','2021-05-04','2021-05-10','2021-05-26',
        '2021-05-29','2021-06-17','2021-06-18','2021-06-20',
        '2021-06-21','2021-06-24','2021-07-02','2021-07-03',
        '2021-07-10','2021-07-29','2021-08-16','2021-08-20',
        '2021-08-23','2021-09-28','2021-09-29','2021-10-01',
        '2021-10-02','2021-10-23','2021-10-27','2021-11-03',
        '2021-11-04','2021-11-20','2021-11-22','2021-11-27',
        '2021-12-04','2021-12-05','2021-12-06','2021-12-13',
        '2021-12-18','2021-12-20','2021-12-22','2021-12-25',
        '2021-12-27','2021-12-28'
    ],

    # Năm 2022 - THÊM NGÀY VÀO ĐÂY
    2022: [
        '2022-01-01','2022-01-05','2022-01-06','2022-01-12',
        '2022-01-21','2022-01-22','2022-01-23','2022-01-28',
        '2022-02-02','2022-02-04','2022-02-16','2022-02-22',
        '2022-02-24','2022-02-25','2022-02-28','2022-03-01',
        '2022-03-03','2022-03-04','2022-03-08','2022-03-24',
        '2022-03-26','2022-04-03','2022-04-04','2022-04-07',
        '2022-04-09','2022-04-23','2022-04-27','2022-05-04',
        '2022-05-05','2022-05-20','2022-05-29','2022-06-05',
        '2022-06-24','2022-07-23','2022-07-25','2022-08-16',
        '2022-09-02','2022-09-04','2022-09-11','2022-10-03',
        '2022-10-11','2022-10-13','2022-10-16','2022-10-24',
        '2022-11-04','2022-11-05','2022-11-12','2022-11-27',
        '2022-11-28','2022-11-30','2022-12-03','2022-12-16',
        '2022-12-20','2022-12-24'
    ],

    # Năm 2023 - THÊM NGÀY VÀO ĐÂY
    2023: [
        '2023-01-01','2023-01-03','2023-01-08','2023-01-15',
        '2023-01-26','2023-01-30','2023-01-31','2023-02-02',
        '2023-02-09','2023-02-11','2023-02-22','2023-02-23',
        '2023-02-25','2023-02-26','2023-02-27','2023-03-06',
        '2023-03-09','2023-03-22','2023-03-26','2023-04-05',
        '2023-04-07','2023-04-18','2023-04-23','2023-05-02',
        '2023-05-03','2023-05-09','2023-05-16','2023-05-26',
        '2023-05-29','2023-05-30','2023-06-01','2023-06-03',
        '2023-06-16','2023-06-20','2023-06-21','2023-06-25',
        '2023-07-05','2023-07-06','2023-07-26','2023-08-10',
        '2023-08-11','2023-08-13','2023-08-21','2023-09-20',
        '2023-09-21','2023-09-24','2023-10-09','2023-11-01',
        '2023-11-15','2023-12-26','2023-12-28'
    ],
    # Nam 2024 - them vao day
    2024: [
        '2024-01-02','2024-01-27','2024-02-03','2024-02-05',
        '2024-02-10','2024-02-28','2024-03-08','2024-03-22',
        '2024-04-09','2024-04-16','2024-04-23','2024-05-25',
        '2024-06-12','2024-07-05','2024-08-06','2024-08-09',
        '2024-08-15','2024-09-16','2024-10-05','2024-10-06',
        '2024-10-13','2024-10-14','2024-10-18','2024-10-21',
        '2024-11-10','2024-11-17','2024-11-19','2024-11-23',
        '2024-12-01','2024-12-05','2024-12-21'
    ],
}

# BƯỚC 7: CHẠY XỬ LÝ

In [ ]:
# Kiểm tra năm có trong danh sách không
if YEAR not in dates_by_year:
    print(f'❌ LỖI: Chưa có dữ liệu cho năm {YEAR}!')
    print(f'💡 Vui lòng thêm danh sách ngày vào dates_by_year[{YEAR}]')
    raise ValueError(f'Không tìm thấy dữ liệu cho năm {YEAR}')

dates_to_process = dates_by_year[YEAR]

if len(dates_to_process) == 0:
    print(f'❌ LỖI: Danh sách ngày cho năm {YEAR} đang trống!')
    print(f'💡 Vui lòng thêm các ngày vào dates_by_year[{YEAR}]')
    raise ValueError(f'Danh sách ngày trống cho năm {YEAR}')

# Tạo tên folder tự động
folder_name = f'{FOLDER_BASE_NAME}_{YEAR}'

print(f'📅 Năm: {YEAR}')
print(f'📁 Folder: {folder_name}')
print(f'📊 Số ngày: {len(dates_to_process)}')
print(f'🔗 Theo dõi: https://code.earthengine.google.com/tasks')
print('\n⏳ Bắt đầu tạo tasks...\n')

successful = 0
failed = 0
start_time = time.time()

for date_str in dates_to_process:
    if process_date(date_str, folder_name):
        successful += 1
    else:
        failed += 1
    time.sleep(0.5)

end_time = time.time()
duration = end_time - start_time

# ============================================================
# KẾT QUẢ
# ============================================================

print(f'✓ Thành công: {successful}/{len(dates_to_process)}')
print(f'✗ Thất bại: {failed}/{len(dates_to_process)}')
print(f'⏱ Thời gian: {duration:.1f} giây')

time.sleep(2)
status = check_tasks_status()

print(f'📋 Chờ xử lý: {status["READY"]}')
print(f'⚙️  Đang chạy: {status["RUNNING"]}')
print(f'✅ Hoàn thành: {status["COMPLETED"]}')
print(f'❌ Thất bại: {status["FAILED"]}')
print(f'🚫 Đã hủy: {status["CANCELLED"]}')
print('1. 🔗 Kiểm tra: https://code.earthengine.google.com/tasks')
print(f'2. 📁 File lưu tại: Google Drive/{folder_name}/')
print(f'\n🎉 ĐÃ TẠO XONG {successful} TASKS CHO NĂM {YEAR}!')
